# Z3rno + LangChain Integration

This notebook demonstrates how to use Z3rno as a memory backend for LangChain applications.

**Prerequisites:**
- A running z3rno-server instance (see [self-hosting docs](../self-hosting))
- An OpenAI API key (for the LLM)
- Z3rno API key

In [ ]:
# Install dependencies
!pip install z3rno[langchain] langchain langchain-openai

## Setup

Configure your Z3rno client and OpenAI credentials.

In [ ]:
import os

# Z3rno configuration
os.environ["Z3RNO_BASE_URL"] = "http://localhost:8000"
os.environ["Z3RNO_API_KEY"] = "z3rno_sk_test_abc123def456"

# OpenAI configuration
os.environ["OPENAI_API_KEY"] = "sk-your-openai-key-here"

## Conversation Memory with Z3rnoChatMessageHistory

Z3rno provides a `ChatMessageHistory` implementation that persists messages
across sessions. Combined with `RunnableWithMessageHistory`, your LangChain
chains automatically retain conversation context.

In [ ]:
from z3rno.integrations.langchain import Z3rnoChatMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# Create the LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

# Define a prompt with message history placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the conversation history to provide contextual responses."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm

In [ ]:
# Factory function that returns a Z3rnoChatMessageHistory for a given session
def get_session_history(session_id: str):
    return Z3rnoChatMessageHistory(
        session_id=session_id,
        base_url="http://localhost:8000",
        api_key="z3rno_sk_test_abc123def456",
        user_id="user_42",
    )

# Wrap the chain with message history
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [ ]:
# First conversation turn
response = chain_with_history.invoke(
    {"input": "My name is Alice and I'm working on a machine learning project."},
    config={"configurable": {"session_id": "session_001"}},
)
print(response.content)

In [ ]:
# Second turn - the assistant remembers the context
response = chain_with_history.invoke(
    {"input": "What libraries would you recommend for my project?"},
    config={"configurable": {"session_id": "session_001"}},
)
print(response.content)

## Memory Persistence Across Sessions

Because Z3rno stores messages server-side, you can resume a conversation
even after restarting your application. The same `session_id` loads the
full history.

In [ ]:
# Simulate a new application session by creating a fresh history object
resumed_history = Z3rnoChatMessageHistory(
    session_id="session_001",
    base_url="http://localhost:8000",
    api_key="z3rno_sk_test_abc123def456",
    user_id="user_42",
)

# All previous messages are still available
print(f"Messages in session: {len(resumed_history.messages)}")
for msg in resumed_history.messages:
    print(f"  [{msg.type}]: {msg.content[:80]}...")

## Retrieval-Augmented Generation with Z3rnoRetriever

Z3rno can also serve as a retriever, enabling semantic search over stored
memories. This is useful for pulling in relevant past context or knowledge.

In [ ]:
from z3rno.integrations.langchain import Z3rnoRetriever
from langchain.chains import RetrievalQA

# Create a Z3rno retriever
retriever = Z3rnoRetriever(
    base_url="http://localhost:8000",
    api_key="z3rno_sk_test_abc123def456",
    user_id="user_42",
    top_k=5,  # number of memories to retrieve
)

# Build a RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

In [ ]:
# Query memories semantically
result = qa_chain.invoke({"query": "What project is Alice working on?"})

print("Answer:", result["result"])
print("\nSource memories:")
for doc in result["source_documents"]:
    print(f"  - {doc.page_content[:100]}")
    print(f"    metadata: {doc.metadata}")

## Summary

With the Z3rno LangChain integration you get:

- **Persistent chat history** via `Z3rnoChatMessageHistory` — messages survive restarts
- **Semantic retrieval** via `Z3rnoRetriever` — search past conversations and stored knowledge
- **Drop-in compatibility** with LangChain's `RunnableWithMessageHistory` and `RetrievalQA`

See the [Z3rno LangChain docs](../integrations/langchain) for full API reference.